<!-- NB_DOC_INTRO_V1 -->
# EDA — Statistiques descriptives & analyses univariees/bivariees

> 📚 **Doc thematique** : [docs/02_EDA.md](docs/02_EDA.md) (EDA & Visualisation)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Phase obligatoire** avant toute modelisation : comprendre les donnees, leur distribution, les manquants, les outliers, les correlations. Ce notebook execute les patterns standards sur le dataset **`tips`** (seaborn, auto-DL).

Couvre : analyses univariees (distrib, normalite, asymetrie), bivariees (quanti×quanti via Pearson/Spearman, quali×quali via chi², quali×quanti via ANOVA/eta²), tableaux croises, Pareto.

## Plan

1. Setup + chargement dataset
2. Statistiques descriptives (`describe`, par type)
3. Univarie quanti : histogramme + KDE + tests normalite
4. Univarie quali : value_counts + barplot + Pareto
5. Manquants (detection + viz)
6. Bivarie quanti×quanti : Pearson, Spearman, scatter, heatmap correlation
7. Bivarie quali×quali : crosstab, chi², Cramer V
8. Bivarie quali×quanti : boxplot, violin, ANOVA, eta²
9. Pieges courants
10. References


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

# Dataset tips (seaborn built-in)
df = sns.load_dataset("tips")
print(f"Shape : {df.shape}")
print(f"Dtypes :\n{df.dtypes}")
df.head()

## 1. Statistiques descriptives

In [ ]:
# Quanti
print("=== Numeriques ===")
print(df.describe().T)

# Categorielles : describe accepte 'category' aussi (et 'object', 'bool')
print("\n=== Categorielles ===")
cat_describe = df.describe(include=["object", "category", "bool"])
if cat_describe.shape[1] > 0:
    print(cat_describe.T)

# Par type
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()
print(f"\nNumeriques  : {num_cols}")
print(f"Categorielles : {cat_cols}")

## 2. Univariee quanti — distribution + tests normalite

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"{col} (n={df[col].notna().sum()})")
plt.tight_layout()
plt.show()

# Tests normalite + skewness + kurtosis
print("=== Tests normalite ===")
for col in num_cols:
    shapiro_stat, shapiro_p = stats.shapiro(df[col].dropna())
    skew = stats.skew(df[col].dropna())
    kurt = stats.kurtosis(df[col].dropna())
    is_normal = shapiro_p > 0.05
    print(f"  {col:10s}: Shapiro p={shapiro_p:.4f} {'(normal)' if is_normal else '(NON normal)'} | "
          f"skew={skew:+.2f} | kurtosis={kurt:+.2f}")

**Interpretation** : `|skew| > 1` = distribution asymetrique forte (transformation log/Box-Cox recommandee avant ML lineaire). `kurtosis > 0` = queues plus epaisses qu'une gaussienne (sensible aux outliers).

## 3. Univariee quali — value_counts + Pareto

In [ ]:
fig, axes = plt.subplots(1, len(cat_cols), figsize=(16, 4))
for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts(normalize=True)
    counts.plot(kind="bar", ax=ax)
    ax.set_title(f"{col}")
    ax.set_ylabel("frequency")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# Cardinalite
print("=== Cardinalite ===")
for col in cat_cols:
    n_unique = df[col].nunique()
    n_total = len(df)
    print(f"  {col:8s}: {n_unique:3d} categories  ({n_unique/n_total:.1%} de la pop)")

### Pareto chart (regle 80/20)

In [ ]:
# Pareto sur day (par fraction des tips totaux)
agg = df.groupby("day", observed=True)["tip"].sum().sort_values(ascending=False)
cum_pct = agg.cumsum() / agg.sum() * 100

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(agg.index.astype(str), agg.values, color="steelblue", edgecolor="black")
ax1.set_ylabel("Tip total ($)", color="steelblue")

ax2 = ax1.twinx()
ax2.plot(agg.index.astype(str), cum_pct.values, color="red", marker="o")
ax2.axhline(80, color="red", ls="--", alpha=0.5)
ax2.set_ylabel("% cumule", color="red")
ax2.set_ylim(0, 105)
plt.title("Pareto chart — tips par jour")
plt.tight_layout()
plt.show()

## 4. Donnees manquantes

In [ ]:
# Detection
missing = df.isna().sum()
print(f"=== Manquants par colonne ===\n{missing[missing > 0]}")
print(f"\nTotal NaN : {df.isna().sum().sum()}")

# Heatmap manquants (pratique sur gros datasets)
fig, ax = plt.subplots(figsize=(8, 3))
sns.heatmap(df.isna().T, cbar=False, cmap="YlOrRd", ax=ax)
ax.set_title("Pattern de NaN (jaune = present, rouge = NaN)")
plt.tight_layout()
plt.show()
print(f"\n(dataset 'tips' n'a pas de NaN — bon eleve)")

## 5. Bivariee quanti × quanti

In [ ]:
# Pearson (lineaire) vs Spearman (monotone)
print("=== Correlations ===")
print(f"\nPearson :\n{df[num_cols].corr(method='pearson').round(3)}")
print(f"\nSpearman :\n{df[num_cols].corr(method='spearman').round(3)}")

In [ ]:
# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(df[num_cols].corr(method="pearson"), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title("Pearson")
sns.heatmap(df[num_cols].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title("Spearman")
plt.tight_layout()
plt.show()

# Pair plot (scatter + KDE diagonale)
sns.pairplot(df, vars=num_cols, hue="sex", height=2.5)
plt.show()

## 6. Bivariee quali × quali — crosstab + chi²

In [ ]:
ct = pd.crosstab(df["sex"], df["day"], margins=True, margins_name="Total")
print(f"=== Crosstab sex × day ===\n{ct}")

# Chi² test d'independance
ct_no_margin = pd.crosstab(df["sex"], df["day"])
chi2, pval, dof, expected = stats.chi2_contingency(ct_no_margin)
print(f"\n=== Chi² test ===")
print(f"  Stat  = {chi2:.4f}")
print(f"  dof   = {dof}")
print(f"  p     = {pval:.4f}")
print(f"  -> H0 (independance) {'rejetee' if pval < 0.05 else 'NON rejetee'}")

# Cramer V (intensite de l'association, [0, 1])
def cramers_v(table):
    chi2 = stats.chi2_contingency(table)[0]
    n = table.values.sum()
    r, c = table.shape
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))
print(f"  Cramer V : {cramers_v(ct_no_margin):.4f}  (0=independance totale, 1=dependance parfaite)")

## 7. Bivariee quali × quanti — boxplot + ANOVA + eta²

In [ ]:
# Boxplot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x="day", y="tip", hue="sex", ax=axes[0])
axes[0].set_title("tip par day et sex")
sns.violinplot(data=df, x="time", y="total_bill", hue="sex", split=True, ax=axes[1])
axes[1].set_title("total_bill par time et sex")
plt.tight_layout()
plt.show()

In [ ]:
# ANOVA : la moyenne des tips varie-t-elle selon le jour ?
groups = [df[df["day"] == d]["tip"].values for d in df["day"].unique()]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"=== ANOVA (tip ~ day) ===")
print(f"  F = {f_stat:.4f}, p = {p_anova:.4f}")
print(f"  -> H0 (moyennes egales) {'rejetee' if p_anova < 0.05 else 'NON rejetee'}")

# eta² (intensite de l'effet, [0, 1])
def eta_squared(groups):
    grand_mean = np.mean(np.concatenate(groups))
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum((x - grand_mean)**2 for g in groups for x in g)
    return ss_between / ss_total

print(f"  eta² : {eta_squared(groups):.4f}  (0=aucun effet, 1=effet parfait)")

## 8. Pieges courants

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Reporter `accuracy` sur classes desequilibrees | Cf. [DS_Metrics_Reference.ipynb](./DS_Metrics_Reference.ipynb) |
| Conclure causalite a partir de correlation | Voir [DS_Causal_Inference.ipynb](./DS_Causal_Inference.ipynb) |
| Pearson sur distrib non-lineaire monotone | Utiliser Spearman ou Kendall |
| Test Shapiro sur N > 5000 | Toujours significatif — utiliser histogramme + QQ-plot |
| Ignorer les valeurs aberrantes | Boxplot, Z-score, IQR (cf. Detection_outliers) |
| p-value comme unique critere | Toujours la taille d'effet (Cramer V, eta², Cohen's d) |
| `df.fillna(0)` aveugle | Comprendre le mecanisme MCAR/MAR/MNAR |
| Standardiser avant le split | Data leakage — Pipeline sklearn |


## References

### Documentation officielle
- scipy.stats : https://docs.scipy.org/doc/scipy/reference/stats.html
- seaborn : https://seaborn.pydata.org/
- pandas describe : https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html

### Voir aussi (collection)
- [EDA_Visualisation_Introduction.ipynb](EDA_Visualisation_Introduction.ipynb)
- [EDA_Analyse_Multivarié.ipynb](EDA_Analyse_Multivarié.ipynb)
- [Détection D_outliers.ipynb](Détection D_outliers.ipynb)
- [DS_Metrics_Reference.ipynb](DS_Metrics_Reference.ipynb)
